# 02. TF-IDF Baseline

Enhanced TF-IDF with handcrafted features + calibration.

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from scipy.sparse import hstack
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

# Target
train_df['target'] = (train_df['winner_model_a'].astype(int) * 0 +
                      train_df['winner_model_b'].astype(int) * 1 +
                      train_df['winner_tie'].astype(int) * 2)

In [3]:
# TF-IDF features
train_text = (train_df['prompt'].fillna('') + ' ' +
               train_df['response_a'].fillna('') + ' ' +
               train_df['response_b'].fillna(''))
test_text = (test_df['prompt'].fillna('') + ' ' +
              test_df['response_a'].fillna('') + ' ' +
              test_df['response_b'].fillna(''))

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(train_text)
X_test_tfidf = vectorizer.transform(test_text)

print(f'TF-IDF shape: {X_train_tfidf.shape}')

TF-IDF shape: (57477, 50000)


In [ ]:
# Combine
X_train = hstack([X_train_tfidf, X_train_hand]).tocsr()
X_test = hstack([X_test_tfidf, X_test_hand]).tocsr()
y_train = train_df['target'].values

print(f'Combined shape: {X_train.shape}')

In [18]:
# Fix: ensure CSR format (kernel may have stale COO from previous run)
X_train = X_train.tocsr() if hasattr(X_train, 'tocsc') else X_train

# CV evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((X_train.shape[0], 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    model = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs')
    model.fit(X_train[train_idx], y_train[train_idx])
    oof_preds[val_idx] = model.predict_proba(X_train[val_idx])
    fold_loss = log_loss(y_train[val_idx], oof_preds[val_idx])
    print(f'Fold {fold+1}: log_loss={fold_loss:.4f}')

oof_loss = log_loss(y_train, oof_preds)
print(f'\nOOF log_loss: {oof_loss:.4f}')

Fold 1: log_loss=1.0737
Fold 2: log_loss=1.0714
Fold 3: log_loss=1.0731
Fold 4: log_loss=1.0722
Fold 5: log_loss=1.0695

OOF log_loss: 1.0720


In [19]:
# Final model + predictions
model = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', multi_class='multinomial')
model.fit(X_train, y_train)
test_preds = model.predict_proba(X_test)

print(f'Test predictions shape: {test_preds.shape}')
print(f'Row sums: {test_preds.sum(axis=1)[:5]}')

TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

In [ ]:
# Save
np.save(OUTPUT_DIR / 'tfidf_oof.npy', oof_preds)
np.save(OUTPUT_DIR / 'tfidf_test.npy', test_preds)
print('Saved TF-IDF predictions')